In [ ]:
# Run this cell first to install required packages (Colab only)
!pip install spatialmath-python -q

# Spatialmath Python Toolbox — Cheatsheet

**EE0849 Introduction to Robotics** | Istanbul Kültür University

This notebook is a quick reference for the [`spatialmath`](https://github.com/petercorke/spatialmath-python) library by Peter Corke.  
It covers the subset of the API used in this course: **SO3** (rotations) and **SE3** (homogeneous transforms).

---

In [1]:
import numpy as np
from spatialmath import SO3, SE3
np.set_printoptions(precision=4, suppress=True)

---
## 1  What is `spatialmath`?

| Concept | Math | Python class |
|---|---|---|
| **Rotation** in 3-D | $R \in SO(3)$, a $3\times3$ orthonormal matrix | `SO3` |
| **Rigid-body pose** in 3-D | $T \in SE(3)$, a $4\times4$ homogeneous matrix | `SE3` |

Both classes wrap a NumPy array but add:
- Operator overloading (`*` = composition, not element-wise multiply)
- Named constructors (`Rx`, `Ry`, `Rz`, `Rt`, `Tx`, …)
- Property accessors (`.R`, `.t`, `.A`)
- Plotting helpers (`.plot()`)

---
## 2  Creating Rotations — `SO3`

### Elementary rotations about X, Y, Z

In [2]:
# Rotation about X-axis by 30°
Rx = SO3.Rx(30, 'deg')
print("SO3.Rx(30°):")
print(Rx)

SO3.Rx(30°):
   1         0         0         
   0         0.866    -0.5       
   0         0.5       0.866     



In [3]:
# Rotation about Y-axis by 45°
Ry = SO3.Ry(45, 'deg')
print("SO3.Ry(45°):")
print(Ry)

SO3.Ry(45°):
   0.7071    0         0.7071    
   0         1         0         
  -0.7071    0         0.7071    



In [4]:
# Rotation about Z-axis by 90°
Rz = SO3.Rz(90, 'deg')
print("SO3.Rz(90°):")
print(Rz)

SO3.Rz(90°):
   0        -1         0         
   1         0         0         
   0         0         1         



### Radians (default) vs Degrees

```python
SO3.Rz(np.pi/2)       # radians (default)
SO3.Rz(90, 'deg')     # degrees — always pass 'deg' as second arg
```

### Composition of rotations

Use the `*` operator. Order matters! $R_{result} = R_1 \cdot R_2$ means "first apply $R_2$, then $R_1$" (right-to-left, like matrix multiplication).

In [5]:
# Rotate 90° about Z, then 45° about Y (of current frame)
R_composed = SO3.Ry(45, 'deg') * SO3.Rz(90, 'deg')
print("Ry(45°) * Rz(90°):")
print(R_composed)

Ry(45°) * Rz(90°):
   0        -0.7071    0.7071    
   1         0         0         
   0         0.7071    0.7071    



### Extract the 3×3 NumPy array

In [6]:
R_matrix = Rz.R          # 3x3 ndarray
print(type(R_matrix))    # <class 'numpy.ndarray'>
print(R_matrix.shape)    # (3, 3)
print(R_matrix)

<class 'numpy.ndarray'>
(3, 3)
[[ 0. -1.  0.]
 [ 1.  0.  0.]
 [ 0.  0.  1.]]


### Inverse of a rotation

For rotations $R^{-1} = R^T$.

In [7]:
R_inv = Rz.inv()
print("Rz(90°) inverse:")
print(R_inv)
print("\nVerify R * R_inv = I:")
print(Rz * R_inv)

Rz(90°) inverse:
   0         1         0         
  -1         0         0         
   0         0         1         


Verify R * R_inv = I:
   1         0         0         
   0         1         0         
   0         0         1         



---
## 3  Creating Homogeneous Transforms — `SE3`

An `SE3` object encodes both **rotation** and **translation** in a single $4\times4$ matrix:

$$
T = \begin{bmatrix} R & p \\ 0 & 1 \end{bmatrix} \in SE(3)
$$

### 3.1  Identity

In [8]:
T_eye = SE3()           # 4x4 identity
print(T_eye)

   1         0         0         0         
   0         1         0         0         
   0         0         1         0         
   0         0         0         1         



### 3.2  Pure translations — `Tx`, `Ty`, `Tz`

These create $T$ with $R = I$ and a translation along one axis.

In [ ]:
T_tx = SE3.Tx(3)        # translate 3 along X
T_ty = SE3.Ty(-1)       # translate -1 along Y
T_tz = SE3.Tz(0.5)      # translate 0.5 along Z

print("SE3.Tx(3):")
print(T_tx)

### 3.3  Pure rotations — `Rx`, `Ry`, `Rz`

These create $T$ with $p = 0$ and a rotation about one axis.  
Same syntax as `SO3.Rx(...)`, but returns an `SE3` (4×4) instead of an `SO3` (3×3).

In [ ]:
T_rz = SE3.Rz(90, 'deg')   # 4x4 with Rz(90°) and zero translation
print("SE3.Rz(90°):")
print(T_rz)

### 3.4  Rotation + Translation — `SE3.Rt(R, t)`

Build a full transform from a $3\times3$ rotation matrix and a translation vector.

⚠️ **Important:** `R` must be a **3×3 NumPy array**, not an `SO3` or `SE3` object.

In [ ]:
# CORRECT — pass .R to extract the 3x3 array
T_full = SE3.Rt(SE3.Rz(90, 'deg').R, [2, 1, 0])
print("SE3.Rt(Rz(90°), [2,1,0]):")
print(T_full)

# WRONG — this will crash:
# SE3.Rt(SE3.Rz(90, 'deg'), [2, 1, 0])   # ❌ TypeError

### 3.5  Build by composition

The most common way to build a full transform: chain elementary operations.

In [ ]:
# "Rotate 90° about Z, then translate 2 along (new) X"
T_chain = SE3.Rz(90, 'deg') * SE3.Tx(2)
print("Rz(90°) * Tx(2):")
print(T_chain)

---
## 4  Properties — Extracting Parts of a Transform

| Property | Returns | Shape | Meaning |
|----------|---------|-------|--------|
| `.A`     | `ndarray` | `(4,4)` for SE3, `(3,3)` for SO3 | Full underlying matrix |
| `.R`     | `ndarray` | `(3,3)` | Rotation sub-matrix |
| `.t`     | `ndarray` | `(3,)` | Translation vector |
| `.n`     | `ndarray` | `(3,)` | X-axis of the rotated frame (1st column of R) |
| `.o`     | `ndarray` | `(3,)` | Y-axis of the rotated frame (2nd column of R) |
| `.a`     | `ndarray` | `(3,)` | Z-axis of the rotated frame (3rd column of R) |

In [ ]:
T = SE3.Rt(SE3.Rz(45, 'deg').R, [3, 1, 2])

print("Full 4x4 matrix (.A):")
print(T.A)
print("\nRotation part (.R):")
print(T.R)
print("\nTranslation part (.t):")
print(T.t)
print("\nX-axis of frame (.n):", T.n)
print("Y-axis of frame (.o):", T.o)
print("Z-axis of frame (.a):", T.a)

---
## 5  Operations

### 5.1  Composition (`*`)

Chaining transforms: ${}^{A}T_{C} = {}^{A}T_{B} \cdot {}^{B}T_{C}$

In [ ]:
T_AB = SE3.Tx(1) * SE3.Rz(90, 'deg')    # Frame B w.r.t. A
T_BC = SE3.Ty(2)                          # Frame C w.r.t. B

T_AC = T_AB * T_BC                        # Frame C w.r.t. A
print("T_AC = T_AB * T_BC:")
print(T_AC)

### 5.2  Inverse (`.inv()`)

If ${}^{A}T_{B}$ describes frame B in frame A, then ${}^{B}T_{A} = ({}^{A}T_{B})^{-1}$.

In [ ]:
T_BA = T_AB.inv()
print("T_AB:")
print(T_AB)
print("\nT_BA = T_AB.inv():")
print(T_BA)
print("\nVerify T_AB * T_BA = I:")
print(T_AB * T_BA)

### 5.3  Transforming a point

Transform a 3-D point from one frame to another: ${}^{A}p = {}^{A}T_{B} \cdot {}^{B}p$

The library automatically handles the homogeneous coordinate (appends 1).

In [ ]:
# Point in frame B
p_B = [1, 0, 0]

# Express it in frame A
p_A = T_AB * p_B
print(f"Point in B: {p_B}")
print(f"Point in A: {p_A}")

---
## 6  Visualization — `.plot()`

Every `SE3` (and `SO3`) object can draw its coordinate frame in a Matplotlib 3-D axis.

In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='3d')

# Plot the world frame
SE3().plot(ax=ax, frame='W', color='gray', length=1)

# Plot a rotated + translated frame
T1 = SE3.Rz(45, 'deg') * SE3.Tx(2)
T1.plot(ax=ax, frame='1', color='blue', length=0.8)

# Plot another frame
T2 = SE3.Tx(1) * SE3.Tz(1.5) * SE3.Rx(90, 'deg')
T2.plot(ax=ax, frame='2', color='red', length=0.8)

ax.set_xlim([-1, 4]); ax.set_ylim([-1, 4]); ax.set_zlim([-1, 3])
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.set_title('Coordinate Frames')
plt.tight_layout()
plt.show()

### `.plot()` keyword arguments

| Keyword | Default | Meaning |
|---------|---------|--------|
| `ax` | current axes | Matplotlib 3-D axes to draw on |
| `frame` | `''` | Label drawn next to the frame origin (e.g. `'W'`, `'1'`) |
| `color` | `'blue'` | Color for all three arrows |
| `length` | `1` | Arrow length (in data units) |

---
## 7  Euler Angles & RPY

### 7.1  Create from Euler angles

In [ ]:
# ZYZ Euler angles (α=30°, β=45°, γ=60°)
R_euler = SO3.Eul(30, 45, 60, unit='deg')   # Rz(α) * Ry(β) * Rz(γ)
print("ZYZ Euler (30°, 45°, 60°):")
print(R_euler)

In [ ]:
# Roll-Pitch-Yaw (XYZ fixed axes)
R_rpy = SO3.RPY(10, 20, 30, unit='deg')     # Rz(yaw) * Ry(pitch) * Rx(roll)
print("RPY (roll=10°, pitch=20°, yaw=30°):")
print(R_rpy)

### 7.2  Extract Euler angles from a rotation

In [ ]:
# Extract ZYZ Euler angles
angles_eul = R_euler.eul(unit='deg')
print(f"ZYZ Euler angles: α={angles_eul[0]:.1f}°, β={angles_eul[1]:.1f}°, γ={angles_eul[2]:.1f}°")

# Extract Roll-Pitch-Yaw
angles_rpy = R_rpy.rpy(unit='deg')
print(f"RPY angles: roll={angles_rpy[0]:.1f}°, pitch={angles_rpy[1]:.1f}°, yaw={angles_rpy[2]:.1f}°")

### 7.3  Euler/RPY with SE3

The same methods exist on `SE3` — they just return/expect a 4×4 matrix.

In [ ]:
T_rpy = SE3.RPY(10, 20, 30, unit='deg')   # SE3 with RPY rotation, zero translation
print("SE3.RPY(10°, 20°, 30°):")
print(T_rpy)

---
## 8  Interoperability with NumPy

### SE3/SO3 → NumPy

In [ ]:
T = SE3.Tx(1) * SE3.Rz(45, 'deg')

# Get the 4x4 matrix
H = T.A                          # np.ndarray, shape (4,4)

# Get rotation and translation separately
R = T.R                          # np.ndarray, shape (3,3)
p = T.t                          # np.ndarray, shape (3,)

print(f"Type of .A: {type(H)}, shape: {H.shape}")
print(f"Type of .R: {type(R)}, shape: {R.shape}")
print(f"Type of .t: {type(p)}, shape: {p.shape}")

### NumPy → SE3/SO3

In [ ]:
# From a 4x4 NumPy matrix
H = np.array([
    [ 0, -1,  0,  3],
    [ 1,  0,  0, -1],
    [ 0,  0,  1,  2],
    [ 0,  0,  0,  1.0]
])
T_from_np = SE3(H)
print("SE3 from NumPy 4x4:")
print(T_from_np)

# From R + t
R = np.eye(3)
t = np.array([1, 2, 3])
T_from_Rt = SE3.Rt(R, t)
print("\nSE3 from R + t:")
print(T_from_Rt)

---
## 9  Common Patterns in This Course

### Pattern 1: Forward Kinematics Chain

${}^{0}T_3 = {}^{0}T_1 \cdot {}^{1}T_2 \cdot {}^{2}T_3$

In [ ]:
T_01 = SE3.Rz(30, 'deg') * SE3.Tx(1)       # link 1
T_12 = SE3.Rz(-45, 'deg') * SE3.Tx(0.8)    # link 2
T_23 = SE3.Rz(60, 'deg') * SE3.Tx(0.5)     # link 3

T_03 = T_01 * T_12 * T_23                   # end-effector in base
print("End-effector pose T_03:")
print(T_03)
print(f"\nEnd-effector position: {T_03.t}")

### Pattern 2: Change of reference frame

Given ${}^{A}p$ (point in frame A), express it in frame B:  
${}^{B}p = {}^{B}T_A \cdot {}^{A}p = ({}^{A}T_B)^{-1} \cdot {}^{A}p$

In [ ]:
T_AB = SE3.Tx(2) * SE3.Rz(90, 'deg')
p_A = [3, 1, 0]                            # point in frame A

p_B = T_AB.inv() * p_A                     # same point in frame B
print(f"Point in A: {p_A}")
print(f"Point in B: {np.round(p_B, 4)}")

### Pattern 3: Verify a transform is valid

In [ ]:
T = SE3.Rz(45, 'deg') * SE3.Tx(1)
R = T.R

# Check: R^T * R should be I
print("R^T * R (should be I):")
print(np.round(R.T @ R, 10))

# Check: det(R) should be +1
print(f"\ndet(R) = {np.linalg.det(R):.6f}")

---
## 10  Quick Reference Card

### Creation

| Code | Description |
|------|-------------|
| `SE3()` | Identity (4×4 eye) |
| `SE3.Tx(d)`, `Ty(d)`, `Tz(d)` | Pure translation along one axis |
| `SE3.Rx(θ, 'deg')`, `Ry(θ, 'deg')`, `Rz(θ, 'deg')` | Pure rotation about one axis |
| `SE3.Rt(R_3x3, [x,y,z])` | From 3×3 rotation + translation |
| `SE3(H_4x4)` | From a NumPy 4×4 matrix |
| `SE3.RPY(r, p, y, unit='deg')` | From roll-pitch-yaw angles |
| `SE3.Eul(α, β, γ, unit='deg')` | From ZYZ Euler angles |
| `SO3.Rx(θ, 'deg')`, `Ry`, `Rz` | 3×3 rotation matrix |
| `SO3.RPY(...)`, `SO3.Eul(...)` | 3×3 from angle conventions |

### Properties

| Code | Returns |
|------|--------|
| `T.A` | Full matrix as ndarray |
| `T.R` | 3×3 rotation part |
| `T.t` | Translation vector (3,) |
| `T.n`, `T.o`, `T.a` | Column vectors of R (x, y, z axes) |
| `T.eul(unit='deg')` | ZYZ Euler angles |
| `T.rpy(unit='deg')` | Roll-pitch-yaw angles |

### Operations

| Code | Meaning |
|------|--------|
| `T1 * T2` | Composition: $T_1 \cdot T_2$ |
| `T * [x,y,z]` | Transform a point |
| `T.inv()` | Inverse transform |
| `T.plot(ax=ax, frame='A', color='b', length=1)` | Draw coordinate frame |

### Gotchas

| Mistake | Fix |
|---------|----|
| `SE3.Rt(SE3.Rz(90,'deg'), t)` | `SE3.Rt(SE3.Rz(90,'deg')`.R`, t)` — pass `.R` not the object |
| `SE3.Rz(90)` gives wrong result | Add `'deg'` — default is radians: `SE3.Rz(90, 'deg')` |
| `T1 * T2` order confusion | Read right-to-left: `T_AB * T_BC` = first B→C, then A→B |

---

*Cheatsheet for EE0849 Introduction to Robotics — based on [spatialmath-python](https://github.com/petercorke/spatialmath-python) v1.x by Peter Corke.*